In [ ]:
output_dir = "../hendrix_outputs"

In [ ]:
import sys
sys.path.append("..")

import os
import pandas as pd
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from pprint import pp
import ipywidgets as widgets
from IPython.display import display, clear_output

from src.utils.visualize import mask_to_alpha_image

In [ ]:
run_groups = os.listdir(output_dir)
run_groups.sort()

def make_run_selectors(change):
    with run_selectors_out:
        all_runs = {group: os.listdir(os.path.join(output_dir, group)) for group in change["new"]}
        for runs in all_runs.values():
            runs.sort()
        run_selectors.clear()
        for group, runs in all_runs.items():
            run_selectors[group] = widgets.SelectMultiple(options=runs, value=runs, description=f"{group}:")
        run_selectors_out.clear_output(wait=True)
        for selector in run_selectors.values():
            selector.layout.height = f"{8+len(selector.options)*19}px"
        display(widgets.HBox(list(run_selectors.values())))

run_group_selector = widgets.SelectMultiple(options=run_groups, description="Select run groups:")
run_group_selector.layout.height = f"{20+len(run_groups)*19}px"
run_selectors = {}
run_group_selector.observe(make_run_selectors, names="value")

display(run_group_selector)
run_selectors_out = widgets.Output()
run_selectors_out

In [ ]:
def flatten_config(conf):
    dict_ = OmegaConf.to_container(conf, resolve=True)
    return flatten_config_helper(dict_)

def flatten_config_helper(original, prefix=""):
    result = {}
    for key, val in original.items():
        if isinstance(val, dict):
            result = result | flatten_config_helper(val, prefix + key + ".")
        elif isinstance(val, list):
            if len(val)>0 and isinstance(val[0], dict): # flatten dicts to make hashable
                for dict_ in val:
                    result = result | flatten_config_helper(dict_, prefix + key + ".")
            else: # convert to tuple to make it hashable
                result[prefix + key] = tuple(val)
        else:
            result[prefix + key] = val
    return result

def config_diff(configs):
    sets = [set(conf.items()) for conf in configs]
    first_set = sets[0]
    set_diff = set()
    for set_ in sets[1:]:
        set_diff = set_diff.union(first_set ^ set_)
    return list(dict(set_diff).keys())

runs = []
run_paths = []
for group, selector in run_selectors.items():
    runs.extend([run for run in selector.value])
    run_paths.extend([os.path.join(output_dir, group, run) for run in selector.value])

config_list = [OmegaConf.load(os.path.join(path, ".hydra/config.yaml")) for path in run_paths]
for i, conf in enumerate(config_list):
    conf.run_name = runs[i]

if len(config_list) > 1:
    flattened_configs = [flatten_config(conf) for conf in config_list]
    config_diff_keys = config_diff(flattened_configs)
    conf_selector = widgets.SelectMultiple(
        options=config_diff_keys,
        value=config_diff_keys,
        description='Select config parameters to include:',
        )
else:
    flattened_configs = [flatten_config(config_list[0])]
    config_diff_keys = []
    conf_selector = widgets.SelectMultiple(
        options=[],
        value=[],
        description='Select config parameters to include:',
        )
conf_selector.layout.height = f"{8+len(conf_selector.options)*19}px"
display(conf_selector)

In [ ]:
shared_config = {key: val for key, val in flattened_configs[0].items() if key not in config_diff_keys}
selected_config = [{key: val for key, val in conf.items() if key in conf_selector.value} for conf in flattened_configs]
non_selected_config = [{key: val for key, val in conf.items() if key in config_diff_keys and key not in conf_selector.value} for conf in flattened_configs]
results_list = [pd.read_pickle(os.path.join(path, "results.pickle")) for path in run_paths]
for i, results in enumerate(results_list):
    for name, val in selected_config[i].items():
        results[name] = val
all_results = pd.concat(results_list)
print("Shared config:")
pp(shared_config)
print("\nSelected parameters:")
pp(selected_config)
print("\nNon-selected parameters:")
pp(non_selected_config)

available_metrics = [column_name[10:] for column_name in all_results.columns if column_name[:10] == "all_train_"]

#######################################################################
# Source - https://stackoverflow.com/a/34162576
# Posted by DSM
# Retrieved 2026-04-05, License - CC BY-SA 3.0

def query_df(df, dict_):
    new_dict_ = {}
    for key, val in dict_.items():
        if val != "nan":
            new_dict_[key] = val
    return df.loc[(df[list(new_dict_)] == pd.Series(new_dict_)).all(axis=1)]
    
#######################################################################

In [ ]:
def make_metric_plot(all_results, query_dict, metric, log_scale):
    fig, ax = plt.subplots()
    query_result = query_df(all_results, query_dict)
    if len(query_result) > 1:
        raise ValueError("Not unique combination!")
    elif len(query_result) == 0:
        raise ValueError("No run satisfies combination!")
    else:
        ax.plot(query_result[f"all_train_{metric}"].item(), label=f"train_{metric}")
        ax.plot(query_result[f"all_test_{metric}"].item(), label=f"test_{metric}")
        if log_scale:
            ax.set_yscale("log")
        ax.legend()
        return fig

def make_image_plot(all_results, query_dict, cmap="summer"):
    query_result = query_df(all_results, query_dict)
    if len(query_result) > 1:
        raise ValueError("Not unique combination!")
    elif len(query_result) == 0:
        raise ValueError("No run satisfies combination!")
    else:
        fig, ax = plt.subplots(3, 2, figsize=(12,18))
        for i in range(3):
            ax[i, 0].imshow(query_result["target_image"].item(), cmap=cmap)
            ax[i, 0].set_title("target image")
        
        ax[0, 1].imshow(query_result["target_image"].item(), cmap=cmap)
        mask = query_result["train_mask"].item()
        ax[0, 1].imshow(mask_to_alpha_image(mask, red=0, green=0, blue=0))
        ax[0, 1].set_title("sparse (train)")
    
        ax[1, 1].imshow(query_result["reconstructed_target"].item(), cmap=cmap)
        ax[1, 1].set_title("reconstructed")
    
        ax[2, 1].imshow(query_result["target_image"].item(), cmap=cmap)
        mask = query_result["test_mask"].item()
        ax[2, 1].imshow(mask_to_alpha_image(mask, red=0, green=0, blue=0))
        ax[2, 1].set_title("test")
        for something in ax:
            for this in something:
                this.set_axis_off()
        return fig

def make_plots(change):
    with out:
        out.clear_output(wait=True)
        chosen_options = {list(param_dict)[i]: sel.value for i, sel in enumerate(param_selectors)}
        chosen_options["opencanopy_idx"] = opencanopy_idx_selector.value
        fig_metric = make_metric_plot(all_results=all_results, query_dict=chosen_options, metric=metric_selector.value, log_scale=log_scale_selector.value)
        fig_image = make_image_plot(all_results=all_results, query_dict=chosen_options)
        display(fig_metric)
        display(fig_image)

param_dict = {key: all_results[key].fillna("nan").unique() for key in conf_selector.value}

param_selectors = [widgets.ToggleButtons(options=vals, description=key+":") for key, vals in param_dict.items()]
opencanopy_idx_selector = widgets.Dropdown(
    options=all_results["opencanopy_idx"].unique(),
    description='opencanopy idx:',
)
metric_selector = widgets.Dropdown(
    options=available_metrics,
    value='rescaled_loss',
    description='metric:',
)
log_scale_selector = widgets.Checkbox(
    value=True,
    description='log scale:',
)

for selector in param_selectors:
    selector.observe(make_plots, names="value")
metric_selector.observe(make_plots, names="value")
log_scale_selector.observe(make_plots, names="value")
opencanopy_idx_selector.observe(make_plots, names="value")


for selector in param_selectors:
    display(selector)
print("--------------------------------------------------------------------------------")
display(widgets.HBox([metric_selector, log_scale_selector]))
display(opencanopy_idx_selector)

out = widgets.Output()
out

In [ ]:
# let us assume all combinations of chosen parameters are present
def make_image_grid(opencanopy_idx, x_var, y_var, other_params, flip_x, flip_y, image_name, mask_name, metric, log_scale, color_within, color_by, show_target_reference):
    assert (image_name is None and metric is not None) or (image_name is not None and metric is None)
    assert (not (show_target_reference and metric is not None)), "Can only show target reference for images"
    size = 4
    x_vals = all_results[x_var].dropna().sort_values(ascending=flip_x).unique()
    y_vals = all_results[y_var].dropna().sort_values(ascending=flip_y).unique()
    x_len = len(x_vals)
    y_len = len(y_vals)
    
    min_metric_val = float("inf")
    max_metric_val = float("-inf")
    min_pixel_val = float("inf")
    max_pixel_val = float("-inf")
    for y_val in y_vals:
        for x_val in x_vals:
            query_result = query_df(all_results, {"opencanopy_idx": opencanopy_idx, x_var: x_val, y_var: y_val})
            assert len(query_result) == 1, f"Not unique combination (length: {len(query_result)}) ({x_var}: {x_val}) ({y_var}: {y_val})"
            if metric:
                train_metrics = query_result[f"all_train_{metric}"].item()
                test_metrics = query_result[f"all_test_{metric}"].item()
                train_max = train_metrics.max()
                train_min = train_metrics.min()
                test_max = test_metrics.max()
                test_min = test_metrics.min()
                if train_max > max_metric_val:
                    max_metric_val = train_max
                if train_min < min_metric_val:
                    min_metric_val = train_min
                if test_max > max_metric_val:
                    max_metric_val = test_max
                if test_min < min_metric_val:
                    min_metric_val = test_min
            else:
                img = query_result[image_name].item()
                img_max = img.max()
                img_min = img.min()
                if img_max > max_pixel_val:
                    max_pixel_val = img_max
                if img_min < min_pixel_val:
                    min_pixel_val = img_min

    target_reference = query_result["target_image"].item()
    fig, ax = plt.subplots(y_len, x_len + int(show_target_reference), figsize=(size*(x_len+int(show_target_reference)), size*y_len))
    for i, y_val in enumerate(y_vals):
        if show_target_reference:
            ax[i, 0].imshow(target_reference, vmin=min_pixel_val, vmax=max_pixel_val, cmap="summer")
            ax[i, 0].set_xticks([], [])
            ax[i, 0].set_yticks([], [])
            ax[i, 0].set_title("Target image")
        if color_within == y_var:
            color_within_query_result = query_df(all_results, {"opencanopy_idx": opencanopy_idx, y_var: y_val})[f"test_{color_by}"]
            min_loss = color_within_query_result.min()
            max_loss = color_within_query_result.max()
        for j, x_val in enumerate(x_vals):
            j = j + int(show_target_reference)
            if color_within == x_var:
                color_within_query_result = query_df(all_results, {"opencanopy_idx": opencanopy_idx, x_var: x_val})[f"test_{color_by}"]
                min_loss = color_within_query_result.min()
                max_loss = color_within_query_result.max()
            query_result = query_df(all_results, {"opencanopy_idx": opencanopy_idx, x_var: x_val, y_var: y_val})
            assert len(query_result) == 1, f"Not unique combination (length: {len(query_result)}) ({x_var}: {x_val}) ({y_var}: {y_val})"
            if metric:
                ax[i, j].plot(query_result[f"all_train_{metric}"].item(), label=f"train_{metric}")
                ax[i, j].plot(query_result[f"all_test_{metric}"].item(), label=f"test_{metric}")
                ax[i, j].set_ylim(min_metric_val, max_metric_val)
                if log_scale:
                    ax[i, j].set_yscale("log")
                ax[i, j].legend()
            else:
                img = query_result[image_name].item()
                axesimage = ax[i, j].imshow(img, vmin=min_pixel_val, vmax=max_pixel_val, cmap="summer")
                if mask_name is not None:
                    mask = query_result[mask_name].item()
                    ax[i, j].imshow(mask_to_alpha_image(mask, red=0, green=0, blue=0))
                ax[i, j].set_xticks([], [])
                ax[i, j].set_yticks([], [])
                loss_value = query_result[f"test_{color_by}"].item()
                ax[i, j].patch.set_edgecolor((float((max_loss-loss_value)/(max_loss-min_loss))**1, 0.0, 0.0))
                ax[i, j].patch.set_linewidth(8)
            ax[i, j].set_title(f"{y_var}={y_val}, {x_var}={x_val}")
    #fig.colorbar(axesimage, ax=ax, location="left", shrink=0.5)
    fig.tight_layout()
    if show_target_reference:
        reference_x1, _, width, _ = ax[0, 0].get_position().bounds
        reference_x2, _, _, _ = ax[0, 1].get_position().bounds
        line_x = (reference_x1 + width + reference_x2) / 2
        ax[0, 0].plot([line_x, line_x], [0, 1], color='black', lw=5, transform=fig.transFigure, clip_on=False)
    return fig

def update_image_grid(update):
    with out2:
        out2.clear_output(wait=True)
        match image_selector.value:
            case "target":
                image_column = "target_image"
                mask_column = None
                metric = None
            case "sparse":
                image_column = "target_image"
                mask_column = "train_mask"
                metric = None
            case "reconstructed":
                image_column = "reconstructed_target"
                mask_column = None
                metric = None
            case "test_grid":
                image_column = "target_image"
                mask_column = "test_mask"
                metric = None
            case "metric":
                image_column = None
                mask_column = None
                metric = metric_selector_grid.value
        grid_fig = make_image_grid(opencanopy_idx_selector_grid.value, x_var_selector.value, y_var_selector.value, {}, x_flip.value, y_flip.value, image_column, mask_column, metric, log_scale_selector_grid.value, color_within_selector.value, color_by_selector.value, show_target_reference_selector.value)
        display(grid_fig)

opencanopy_idx_selector_grid = widgets.ToggleButtons(
    options=all_results["opencanopy_idx"].unique(),
    description='opencanopy idx:',
)
image_selector = widgets.ToggleButtons(
    options=["target", "sparse", "reconstructed", "test_grid", "metric"],
    description="show:"
)
metric_selector_grid = widgets.Dropdown(
    options=available_metrics,
    description="metric:"
)
log_scale_selector_grid = widgets.Checkbox(
    description="log scale"
)
x_var_selector = widgets.ToggleButtons(
    options=list(conf_selector.value),
    description="x_var:"
)
y_var_selector = widgets.ToggleButtons(
    options=list(conf_selector.value),
    description="y_var:"
)
x_flip = widgets.Checkbox(
    description="flip x-axis:"
)
y_flip = widgets.Checkbox(
    description="flip y-axis:"
)
color_within_selector = widgets.ToggleButtons(
    options=list(conf_selector.value),
    description="color within:"
)
color_by_selector = widgets.Dropdown(
    options=available_metrics,
    value="rescaled_loss",
    description="color by:"
)
show_target_reference_selector = widgets.Checkbox(
    description="show target reference:"
)

opencanopy_idx_selector_grid.observe(update_image_grid, names="value")
image_selector.observe(update_image_grid, names="value")
x_var_selector.observe(update_image_grid, names="value")
y_var_selector.observe(update_image_grid, names="value")
x_flip.observe(update_image_grid, names="value")
y_flip.observe(update_image_grid, names="value")
metric_selector_grid.observe(update_image_grid, names="value")
log_scale_selector_grid.observe(update_image_grid, names="value")
color_within_selector.observe(update_image_grid, names="value")
color_by_selector.observe(update_image_grid, names="value")
show_target_reference_selector.observe(update_image_grid, names="value")

display(opencanopy_idx_selector_grid)
display(widgets.HBox([x_var_selector, x_flip]))
display(widgets.HBox([y_var_selector, y_flip]))
display(widgets.HBox([image_selector, metric_selector_grid, log_scale_selector_grid]))
display(widgets.HBox([color_within_selector, color_by_selector]))
display(show_target_reference_selector)

out2 = widgets.Output()
out2

In [ ]:
metric_columns = [f"train_{metric}" for metric in available_metrics] + [f"test_{metric}" for metric in available_metrics]
relevant_columns = list(conf_selector.value) + ["opencanopy_idx"] + metric_columns
final_metrics = all_results[relevant_columns]
final_metrics

In [ ]:
final_metrics.groupby(["data.sparsity", "model.scale"]).mean().sort_values(["data.sparsity", "test_RMSE"])[["train_RMSE", "test_RMSE"]]
#final_metrics[~(final_metrics.opencanopy_idx == 19566)].groupby(["model.scale", "data.normalization_max"]).mean().sort_values(["data.normalization_max", "test_RMSE"], ascending=True).groupby("data.normalization_max").head(1)[["test_PSNR", "test_SSIM", "test_loss", "test_MSE", "test_RMSE"]]

In [ ]:
# Todo
# Colorbars
# Fixed pixel ranges in single run inspection (custom ranges?)
# Make it possible to select values for configs not to be used in grid (model, when comparing ff and siren)
# Make color_by work with metrics where lower != better.